# 🧮 Understanding life history dynamics in structured populations with a stage-within-age matrix model

This noteboook implements a structured population model by Diamond *et al*. (1999, 2000)* of a fish called the Atlantic croaker, *Micropogonias undulatus* (for additional information on this species see <a href="http://www.tpwd.state.tx.us/huntwild/wild/species/croaker/.">http://www.tpwd.state.tx.us/huntwild/wild/species/croaker/</a> ). 

![alt](http://www.tpwd.state.tx.us/huntwild/wild/images/fish/acroaker.jpg)

An image of an Atlantic croaker from the [Texas Parks and Wildlife Department](http://www.tpwd.state.tx.us/huntwild/wild/species/croaker).

Key aspects of Atlantic croaker demography, and how Diamond *et al*. formulated their mode of it, are discussed in the [introduction to this Unit](images/Croaker.md) and associated pages.  

***Citations**:  
Sandra L. Diamond, Larry B. Crowder, Lindsay G. Cowell (1999). *Catch and Bycatch: The Qualitative Effects of Fisheries on Population Vital Rates of Atlantic Croaker.* Transactions of the American Fisheries Society; 128(6):1085–1105  
Sandra L Diamond; Lindsay G Cowell; Larry B Crowder (2000). *Population effects of shrimp trawl bycatch on Atlantic croaker.* Canadian Journal of Fisheries and Aquatic Sciences; 57( 10):2010-21.


In [ ]:
# Load modules and set graphics environment
from math import *
import numpy as np
from matplotlib import pyplot as plt
plt.ion();
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
global A
from croaker import LesliePars, CroakerPopModel

## Exploration of the stage-within-age model: Model Inputs
The grid of input boxes below enable you to experiment with population trajectories for for the Atlantic croaker according to Diamond *et al*.'s stage-within-age model. 
The default demographic parameters are those calculated by Diamond *et al*. (1999, 2000) for the Atlantic croaker population in the Gulf of Mexico.

Three types of parameters appear below:
- **Simulation parameters**:
     - The rightmost column ($p_{egg}, p_1, p_2,\dots$) determines the *initial population* in each Year Class (including eggs) for a population simulation.
     - $N_{years}$ is the *number of years* to be simulated.
- **Demographic parameters**:
     - The first column specifies the *duration* of each of the larval and juvenile stages: $T_{egg},T_{YSL}, \dots$
     - The second column specifies the *mortality rate* of each of the larval and juvenile stages: $\mu_{egg},\mu_{YSL}, \dots$ in $days^{-1}$, $\mu_{adult}$ in $years^{-1}$.
     - The third column specifies the *fecundity* of adults in Year Classes $1, 2, \dots$
- **Evolutionary parameters**:
     - The bottom input box in the first column is specifies the growth rate, $\alpha$, during the Ocean Larva stage (see the discussion in [this derivation](./CroakerEIF.md)).
     - The text box above that specifies the Egg Investment Factor, $EIF$.  
       The default value ($EIF=1$) represents the observed egg size, $EIF<1$ implies more numerous but smaller eggs, and $EIF>1$ implies fewer but larger eggs.

## Exploration of the stage-within-age model: Model Outputs
This implementation of the model produces four types of graphical output, in addition to textual numerical data:
- **Age distribution**:

     This plot shows the fraction of the croaker population in each year class.
     Note that the vertical axis is on a log10 scale.
     - The initial age distribution (set by you in the top input grid) is plotted in green.
     - The estimated stable age distribution derived analytically from the Leslie matrix is plotted in blue.
     - The actual final age distribution (from the full stage-within-age model) at the end of the simulation is plotted in black.
- **Population time series**:
 
     This plot shows total population of Atlantic croakers, as a function of years from the start of the simulation.
    - The population trajectory from the full model is shown in green.
    - An analytical approximation to this model, in which it is assumed that the population always matches the stable age distribution, is shown in blue.

- **Larval demography**:

    This plot shows the cummulative probability of survival through the successive larval stages.

    As in Diamond *et al*.'s paper, time units for larval demography are days (not years, as for the time series plot).
    Note the vertical axis is on a log10 scale.
- **Elasticities**:

    This plot shows the elasticities for each demographic parameter in the simulation.

    Elasticities are a metric of sensitivity of long-term population increase or decrease to relatively small changes in parameters:
    - A small positive elasticity implies that increase of the corresponding parameter will slightly raise the long-term population trajectory.
    - A large negative elasticity implies an increase in the corresponding parameter will substantially decrease the long-term population trajectory, and so forth.

In [ ]:
# Instantiate a GUI to modify parameters. 
#  Initial parameters from Diamond et al. (2000) for Atlantic croakers in the Gulf of Mexico:
T_egg=widgets.FloatText(value=2.,description = r"$T_{EGG}$")
T_ysl=widgets.FloatText(value=4.,description = r"$T_{YSL}$")
T_ol=widgets.FloatText(value=45.,description = r"$T_{OL}$")
T_el=widgets.FloatText(value=58.5,description = r"$T_{EL}$")
T_ej=widgets.FloatText(value=75.,description = r"$T_{EJ}$")
T_lj=widgets.FloatText(value=180.,description = r"$T_{LJ}$")
EIF=widgets.FloatText(value=1,description = r"$EIF$")

mu_egg=widgets.FloatText(value=0.4984,description = r"$\mu_{EGG}$")
mu_ysl=widgets.FloatText(value=0.1645,description = r"$\mu_{YSL}$")
mu_ol=widgets.FloatText(value=0.09,description = r"$\mu_{OL}$")
mu_el=widgets.FloatText(value=0.0591,description = r"$\mu_{EL}$")
mu_ej=widgets.FloatText(value=0.021,description = r"$\mu_{EJ}$")
mu_lj=widgets.FloatText(value=0.009,description = r"$\mu_{LJ}$")
mu_adult=widgets.FloatText(value=0.85,description = r"$\mu_{adult}$")

F1=widgets.FloatText(value=0.10,description = r"$F_1$")
F2=widgets.FloatText(value=59581.,description = r"$F_2$")
F3=widgets.FloatText(value=76456.,description = r"$F_3$")
F4=widgets.FloatText(value=93690.,description = r"$F_4$")
F5=widgets.FloatText(value=106614.,description = r"$F_5$")
F6=widgets.FloatText(value=124565.,description = r"$F_6$")
F7=widgets.FloatText(value=130310.,description = r"$F_7$")
alpha=widgets.FloatText(value=0.1,description = r"$\alpha$")

p_egg=widgets.FloatText(value=1.,description = r"$p_{egg}$")
p1=widgets.FloatText(value=1.e-12,description = r"$p_1$")
p2=widgets.FloatText(value=1.e-12,description = r"$p_2$")
p3=widgets.FloatText(value=1.e-12,description = r"$p_3$")
p4=widgets.FloatText(value=1.e-12,description = r"$p_4$")
p5=widgets.FloatText(value=1.e-12,description = r"$p_5$")
p6=widgets.FloatText(value=1.e-12,description = r"$p_6$")
p7=widgets.FloatText(value=1.e-12,description = r"$p_7$")
N_years=widgets.IntText(value=7,description = r"$N_{years}$")

ui1 = widgets.VBox([T_egg,T_ysl,T_ol,T_el,T_ej,T_lj,EIF,alpha])
ui2 = widgets.VBox([mu_egg,mu_ysl,mu_ol,mu_el,mu_ej,mu_lj,mu_adult,N_years])
ui3 = widgets.VBox([F1,F2,F3,F4,F5,F6,F7])
ui4 = widgets.VBox([p_egg,p1,p2,p3,p4,p5,p6,p7])
uiC = widgets.HBox([ui1,ui2,ui3,ui4])
    
out = widgets.interactive_output(CroakerPopModel,{'T_egg':T_egg,'T_ysl':T_ysl,'T_ol':T_ol,'T_el':T_el,'T_ej':T_ej,'T_lj':T_lj,
                 'mu_egg':mu_egg,'mu_ysl':mu_ysl,'mu_ol':mu_ol,'mu_el':mu_el,'mu_ej':mu_ej,'mu_lj':mu_lj,
                 'mu_adult':mu_adult,
                    'F1':F1,'F2':F2,'F3':F3,'F4':F4,'F5':F5,'F6':F6,'F7':F7,
                    'p_egg':p_egg,'p1':p1,'p2':p2,'p3':p3,'p4':p4,'p5':p5,'p6':p6,'p7':p7,
                    'N_years':N_years,'alpha':alpha,'EIF':EIF})
display(uiC,out)

:::{figure} #cr_1
:placeholder: ./images/CR_1.png
:align: left
:::
:::{figure} #cr_2
:placeholder: ./images/CR_2.png
:align: left
:::